# Vuln2Action - CAPEC-conditioned flow pilot

Operational variant: condition T5 on the selected CAPEC name/id plus the CVE
description. This is not the paper's CVE-only input, but it tests whether the
current 169-flow dataset can generate usable flows when the CAPEC label is
available from the pipeline.

- target_steps, 1 epoch first for a quick signal
- max target 384
- batch 8/GPU, grad accum 4
- warmup 0.1
- evaluate 1,000 validation rows


In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__)
print("cuda devices:", torch.cuda.device_count())


In [ ]:
!rm -rf /kaggle/working/Vuln2Action-Lab
!git clone --branch feature/paper-reproduction https://github.com/Hoang-Minh-Hieu-2806/Vuln2Action-Lab.git /kaggle/working/Vuln2Action-Lab
%cd /kaggle/working/Vuln2Action-Lab
!pip install -q -e ".[train]"


In [ ]:
from pathlib import Path
import glob, os
hits=sorted(glob.glob("/kaggle/input/**/flow_train.jsonl", recursive=True))
print("flow_train hits:", hits)
if not hits: raise FileNotFoundError("flow_train.jsonl not found")
FLOW_DIR=Path(hits[0]).parent
os.environ["FLOW_DIR"]=str(FLOW_DIR)
os.environ["TOKENIZERS_PARALLELISM"]="false"
print("FLOW_DIR:", FLOW_DIR)


In [ ]:
import os, sys, subprocess
cmd=[
 sys.executable, "-m", "torch.distributed.run", "--nproc_per_node=2",
 "scripts/train_t5.py",
 "--dataset", os.environ["FLOW_DIR"],
 "--output-dir", "/kaggle/working/t5_stage1_capec_conditioned",
 "--model-name", "t5-base",
 "--target-field", "target_steps",
 "--include-capec",
 "--max-input-length", "256",
 "--max-target-length", "384",
 "--per-device-batch-size", "8",
 "--gradient-accumulation-steps", "4",
 "--learning-rate", "5e-5",
 "--warmup-ratio", "0.1",
 "--num-train-epochs", "1",
 "--save-steps", "693",
 "--eval-steps", "693",
 "--logging-steps", "100",
 "--save-total-limit", "2",
 "--no-load-best-model-at-end",
 "--fp16",
 "--gradient-checkpointing",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True, env=os.environ.copy())


In [ ]:
from pathlib import Path
import os, sys, subprocess
base=Path("/kaggle/working/t5_stage1_capec_conditioned")
final=base/"final"
ckpts=sorted(base.glob("checkpoint-*"), key=lambda p:int(p.name.split("-")[-1]))
print("final exists:", final.exists())
print("checkpoints:", [p.name for p in ckpts])
CKPT=final if final.exists() else (ckpts[-1] if ckpts else None)
if CKPT is None: raise FileNotFoundError("no checkpoint")
print("CKPT:", CKPT)
cmd=[
 sys.executable, "scripts/evaluate_t5.py",
 "--checkpoint", str(CKPT),
 "--dataset", os.environ["FLOW_DIR"],
 "--split", "validation",
 "--target-field", "target_steps",
 "--include-capec",
 "--output-dir", "/kaggle/working/t5_stage1_capec_conditioned_eval",
 "--max-input-length", "256",
 "--max-target-length", "384",
 "--batch-size", "4",
 "--num-beams", "4",
 "--limit", "1000",
 "--fp16",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True, env=os.environ.copy())


In [ ]:
import json
from pathlib import Path
out=Path("/kaggle/working/t5_stage1_capec_conditioned_eval")
metrics=json.loads((out/"metrics.json").read_text(encoding="utf-8"))
print(json.dumps(metrics, indent=2, ensure_ascii=False))
for i,line in enumerate((out/"predictions.jsonl").open(encoding="utf-8")):
    if i>=3: break
    row=json.loads(line)
    print(f"\n[{i}] {row['cve_id']}")
    print("REF:", row["reference"][:400])
    print("PRED:", row["prediction"][:400])
